# Gradient Descent — Implementations

Each optimizer twice more: once on tensors with the arithmetic unchanged, once through `torch.optim` to show the built-in is the same update. The equivalence fixtures use the same literal quadratic in every lane, and `tol=0.0` so no lane breaks out of the loop a step earlier than another.

## 02_gradient_descent

Fixed steps downhill.

### torch

The identical loop on tensors. **What torch adds here:** nothing yet — that is the point of starting with it.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Same loop as NumPy: the tensor is the only thing that changed.
# 2. clone() where NumPy copies — a tensor slice is a view, same trap as NumPy.
# 3. Keep float64; the point of this lane is agreeing with the NumPy one.


def gd_scratch(grad_fn, x0, lr=0.01, max_iter=1000, tol=1e-6):
    """Vanilla gradient descent on tensors — the algorithm is unchanged."""
    x = torch.as_tensor(x0, dtype=torch.float64).clone()
    history = [x.clone()]
    for _ in range(max_iter):
        g = grad_fn(x)
        x_new = x - lr * g
        history.append(x_new.clone())
        if torch.linalg.norm(x_new - x) < tol:
            break
        x = x_new
    return x_new, history


In [ ]:
# exports: x_star, n_steps
_A_eq = torch.tensor([[3.0, 0.5], [0.5, 1.0]], dtype=torch.float64)
_b_eq = torch.tensor([1.0, -2.0], dtype=torch.float64)
def _grad_eq(w):
    return _A_eq @ w - _b_eq
_x0_eq = torch.zeros(2, dtype=torch.float64)

x_star, _hist_eq = gd_scratch(_grad_eq, _x0_eq, lr=0.2, max_iter=300, tol=0.0)
n_steps = len(_hist_eq)
print("x* =", x_star, "in", n_steps - 1, "steps")


In [ ]:
# The optimum of (1/2)wᵀAw − bᵀw is the solution of Aw = b.
_x_opt = torch.linalg.solve(_A_eq, _b_eq)
assert torch.allclose(x_star, _x_opt, atol=1e-6), "descent should reach the closed-form optimum"
_dists = torch.tensor([float(torch.linalg.norm(h - _x_opt)) for h in _hist_eq])
assert bool(torch.all(_dists[1:] <= _dists[:-1] + 1e-12)), "each step moves toward the optimum"


### library

`torch.optim.SGD` without momentum is exactly `x -= lr * g`. **What the library adds:** an interface, not an algorithm.

In [ ]:
import numpy as np
import torch

# hints:
# 1. optim.SGD with no momentum is exactly x -= lr * grad — nothing more.
# 2. Set p.grad yourself before step(); the optimiser only reads it.
# 3. step() mutates in place, so record a detached clone into the history.


def gd_scratch(grad_fn, x0, lr=0.01, max_iter=1000, tol=1e-6):
    """The same loop driven through torch.optim.SGD, to show it hides nothing."""
    x = torch.as_tensor(x0, dtype=torch.float64).clone().requires_grad_(True)
    opt = torch.optim.SGD([x], lr=lr)
    history = [x.detach().clone()]
    for _ in range(max_iter):
        opt.zero_grad()
        x.grad = grad_fn(x.detach())
        opt.step()
        history.append(x.detach().clone())
        if torch.linalg.norm(history[-1] - history[-2]) < tol:
            break
    return x.detach(), history


In [ ]:
# exports: x_star, n_steps
_A_eq = torch.tensor([[3.0, 0.5], [0.5, 1.0]], dtype=torch.float64)
_b_eq = torch.tensor([1.0, -2.0], dtype=torch.float64)
def _grad_eq(w):
    return _A_eq @ w - _b_eq
_x0_eq = torch.zeros(2, dtype=torch.float64)

x_star, _hist_eq = gd_scratch(_grad_eq, _x0_eq, lr=0.2, max_iter=300, tol=0.0)
n_steps = len(_hist_eq)
print("x* =", x_star, "in", n_steps - 1, "steps")


In [ ]:
_x_opt = torch.linalg.solve(_A_eq, _b_eq)
assert torch.allclose(x_star, _x_opt, atol=1e-6), "optim.SGD lands on the same optimum"


## 02_momentum

A velocity that remembers.

### torch

Same heavy-ball convention as the notebook: `v = βv − lr·g, x += v`.

In [ ]:
import numpy as np
import torch

# hints:
# 1. The notebook's convention is v = beta*v - lr*g, then x = x + v.
# 2. The velocity starts at zeros, like the parameters' shadow.
# 3. beta=0 must reduce to vanilla GD — a free sanity check.


def gd_momentum_scratch(grad_fn, x0, lr=0.01, beta=0.9, max_iter=1000, tol=1e-6):
    """Polyak heavy-ball on tensors, same convention as the NumPy lane."""
    x = torch.as_tensor(x0, dtype=torch.float64).clone()
    v = torch.zeros_like(x)
    history = [x.clone()]
    for _ in range(max_iter):
        g = grad_fn(x)
        v = beta * v - lr * g
        x_new = x + v
        history.append(x_new.clone())
        if torch.linalg.norm(x_new - x) < tol:
            break
        x = x_new
    return x_new, history


In [ ]:
# exports: x_star, n_steps
_A_eq = torch.tensor([[3.0, 0.5], [0.5, 1.0]], dtype=torch.float64)
_b_eq = torch.tensor([1.0, -2.0], dtype=torch.float64)
def _grad_eq(w):
    return _A_eq @ w - _b_eq
_x0_eq = torch.zeros(2, dtype=torch.float64)

x_star, _hist_eq = gd_momentum_scratch(_grad_eq, _x0_eq, lr=0.05, beta=0.9,
                                       max_iter=300, tol=0.0)
n_steps = len(_hist_eq)


In [ ]:
_x_opt = torch.linalg.solve(_A_eq, _b_eq)
assert torch.allclose(x_star, _x_opt, atol=1e-6)

# Momentum earns its name: after the same number of steps at the same lr,
# the heavy ball sits closer to the optimum than plain GD does.
_x_plain = _x0_eq.clone()
for _ in range(n_steps - 1):
    _x_plain = _x_plain - 0.05 * _grad_eq(_x_plain)
assert float(torch.linalg.norm(x_star - _x_opt)) < float(torch.linalg.norm(_x_plain - _x_opt)), \
    "momentum should be ahead of plain GD at this budget"


### library

`SGD(momentum=β)` keeps `buf = β·buf + g` and steps `x −= lr·buf`; substitute `v = −lr·buf` and it is the notebook's update, verbatim. **Watch `dampening`:** any nonzero value makes it a different method.

In [ ]:
import numpy as np
import torch

# hints:
# 1. SGD(momentum=beta) keeps buf = beta*buf + g and steps x -= lr*buf.
# 2. Substitute v = -lr*buf and it is literally the notebook's update.
# 3. dampening must stay 0, or the buffer scales g by (1-beta) as well.


def gd_momentum_scratch(grad_fn, x0, lr=0.01, beta=0.9, max_iter=1000, tol=1e-6):
    """torch.optim.SGD(momentum=...) — the same heavy ball under a different name."""
    x = torch.as_tensor(x0, dtype=torch.float64).clone().requires_grad_(True)
    opt = torch.optim.SGD([x], lr=lr, momentum=beta)
    history = [x.detach().clone()]
    for _ in range(max_iter):
        opt.zero_grad()
        x.grad = grad_fn(x.detach())
        opt.step()
        history.append(x.detach().clone())
        if torch.linalg.norm(history[-1] - history[-2]) < tol:
            break
    return x.detach(), history


In [ ]:
# exports: x_star, n_steps
_A_eq = torch.tensor([[3.0, 0.5], [0.5, 1.0]], dtype=torch.float64)
_b_eq = torch.tensor([1.0, -2.0], dtype=torch.float64)
def _grad_eq(w):
    return _A_eq @ w - _b_eq
_x0_eq = torch.zeros(2, dtype=torch.float64)

x_star, _hist_eq = gd_momentum_scratch(_grad_eq, _x0_eq, lr=0.05, beta=0.9,
                                       max_iter=300, tol=0.0)
n_steps = len(_hist_eq)


In [ ]:
_x_opt = torch.linalg.solve(_A_eq, _b_eq)
assert torch.allclose(x_star, _x_opt, atol=1e-6), "same optimum through torch.optim"


## 02_adam

Per-parameter step sizes from two running moments.

### torch

The notebook's Adam, line by line, on tensors.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Two running moments: m tracks the gradient, s tracks its square.
# 2. Bias correction divides by (1 - beta^t) — t starts at 1, not 0.
# 3. eps goes outside the square root in this convention; torch agrees.


def adam_scratch(grad_fn, x0, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8,
                 max_iter=1000, tol=1e-6):
    """Adam on tensors, following the notebook line by line."""
    x = torch.as_tensor(x0, dtype=torch.float64).clone()
    m = torch.zeros_like(x)
    s = torch.zeros_like(x)
    history = [x.clone()]
    for t in range(1, max_iter + 1):
        g = grad_fn(x)
        m = beta1 * m + (1 - beta1) * g
        s = beta2 * s + (1 - beta2) * g ** 2
        m_hat = m / (1 - beta1 ** t)
        s_hat = s / (1 - beta2 ** t)
        x_new = x - lr * m_hat / (torch.sqrt(s_hat) + eps)
        history.append(x_new.clone())
        if torch.linalg.norm(x_new - x) < tol:
            break
        x = x_new
    return x_new, history


In [ ]:
# exports: x_star
_A_eq = torch.tensor([[3.0, 0.5], [0.5, 1.0]], dtype=torch.float64)
_b_eq = torch.tensor([1.0, -2.0], dtype=torch.float64)
def _grad_eq(w):
    return _A_eq @ w - _b_eq
_x0_eq = torch.zeros(2, dtype=torch.float64)

x_star, _hist_eq = adam_scratch(_grad_eq, _x0_eq, lr=0.05, max_iter=500, tol=0.0)


In [ ]:
_x_opt = torch.linalg.solve(_A_eq, _b_eq)
assert float(torch.linalg.norm(x_star - _x_opt)) < 1e-3, "500 Adam steps get close on a quadratic"


### library

`torch.optim.Adam` — identical moments, identical bias correction, identical eps placement. The equivalence check against the scratch lane should read ~1e-15, and if it ever does not, a convention moved.

In [ ]:
import numpy as np
import torch

# hints:
# 1. torch.optim.Adam uses the identical update — same moments, same correction.
# 2. If this lane and the scratch one disagree past 1e-12, a convention differs.
# 3. Compare eps placement first; it is the usual culprit between frameworks.


def adam_scratch(grad_fn, x0, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8,
                 max_iter=1000, tol=1e-6):
    """torch.optim.Adam — bit-for-bit the update the notebook derives."""
    x = torch.as_tensor(x0, dtype=torch.float64).clone().requires_grad_(True)
    opt = torch.optim.Adam([x], lr=lr, betas=(beta1, beta2), eps=eps)
    history = [x.detach().clone()]
    for _ in range(max_iter):
        opt.zero_grad()
        x.grad = grad_fn(x.detach())
        opt.step()
        history.append(x.detach().clone())
        if torch.linalg.norm(history[-1] - history[-2]) < tol:
            break
    return x.detach(), history


In [ ]:
# exports: x_star
_A_eq = torch.tensor([[3.0, 0.5], [0.5, 1.0]], dtype=torch.float64)
_b_eq = torch.tensor([1.0, -2.0], dtype=torch.float64)
def _grad_eq(w):
    return _A_eq @ w - _b_eq
_x0_eq = torch.zeros(2, dtype=torch.float64)

x_star, _hist_eq = adam_scratch(_grad_eq, _x0_eq, lr=0.05, max_iter=500, tol=0.0)


In [ ]:
_x_opt = torch.linalg.solve(_A_eq, _b_eq)
assert float(torch.linalg.norm(x_star - _x_opt)) < 1e-3


## 02_sgd

The full gradient, sampled.

### torch

Mini-batch SGD with tensors doing the arithmetic and NumPy still doing the shuffling — determinism across lanes comes from sharing the permutation, not from hoping.

In [ ]:
import numpy as np
import torch

# hints:
# 1. The shuffling stays NumPy's rng.permutation, so every lane sees the same batches.
# 2. Index a tensor with a NumPy index array — torch accepts it directly.
# 3. The only stochastic thing here is the batch order; the update is plain GD.


def sgd_scratch(grad_batch_fn, x0, n_samples, lr=0.01, max_iter=100,
                batch_size=32, rng=None):
    """Mini-batch SGD on tensors. The randomness is NumPy's, by design:
    two lanes only agree if they draw the same batches in the same order."""
    if rng is None:
        rng = np.random.default_rng(42)
    x = torch.as_tensor(x0, dtype=torch.float64).clone()
    history = [x.clone()]
    for _ in range(max_iter):
        indices = rng.permutation(n_samples)
        for start in range(0, n_samples, batch_size):
            batch = indices[start:start + batch_size]
            g = grad_batch_fn(x, batch)
            x = x - lr * g
        history.append(x.clone())
    return x, history


In [ ]:
# exports: w_star
_rng_eq = np.random.default_rng(5)
X_sgd_eq = np.column_stack([np.ones(64), _rng_eq.normal(size=(64, 2))])
y_sgd_eq = X_sgd_eq @ np.array([1.0, 2.0, -1.0]) + _rng_eq.normal(0, 0.1, size=64)
_Xt_eq = torch.as_tensor(X_sgd_eq)
_yt_eq = torch.as_tensor(y_sgd_eq)
def _grad_batch_eq(w, idx):
    Xb, yb = _Xt_eq[idx], _yt_eq[idx]
    r = Xb @ w - yb
    return (2.0 / len(idx)) * (Xb.T @ r)

w_star, _hist_eq = sgd_scratch(_grad_batch_eq, torch.zeros(3, dtype=torch.float64), 64,
                               lr=0.05, max_iter=30, batch_size=16,
                               rng=np.random.default_rng(7))


In [ ]:
_w_ls = torch.linalg.lstsq(_Xt_eq, _yt_eq.unsqueeze(1)).solution.squeeze(1)
assert float(torch.linalg.norm(w_star - _w_ls)) < 0.05, "SGD hovers near the least-squares solution"
